# Lab 01: Kinematics & The Geometric Workspace

**Computational Sensorimotor Control** | Week 1 | Module 1: The Biological Plant

---

## Overview

In this lab, you will build the `Arm` class — the kinematic skeleton that will be the foundation of your simulator for the entire course. By the end, your arm will be able to:

1. Compute **forward kinematics** (joint angles → hand position)
2. Solve **inverse kinematics** (hand position → joint angles)
3. Compute the **Jacobian matrix** at any configuration
4. Visualize **manipulability ellipsoids** across the workspace

### Key Reference
- Morasso, P. (1981). Spatial control of arm movements. *Experimental Brain Research, 42*(2), 223–227.

---

## Anatomical Conventions (Used Throughout the Course)

All movements are in the **horizontal plane** (no gravity). The shoulder is fixed at the origin.

| Parameter | Symbol | Value | Notes |
|-----------|--------|-------|-------|
| Upper arm length | $l_1$ | 0.34 m | Gribble et al. (1998) |
| Forearm length | $l_2$ | 0.46 m | Gribble et al. (1998) |
| Shoulder angle | $\theta_1$ | 0° – 140° | 0° = arm abducted to side; positive = horizontal flexion (across chest) |
| Elbow angle | $\theta_2$ | 0° – 145° | 0° = fully extended; positive = flexion |

The workspace is a **semicircular crescent** in front of the body, not a full annulus.

In [ ]:
# === Setup ===
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Joint limits (radians) — used throughout the course
THETA1_MIN, THETA1_MAX = np.radians(0), np.radians(140)
THETA2_MIN, THETA2_MAX = np.radians(0), np.radians(145)

print('Setup complete.')
print(f'Shoulder range: {np.degrees(THETA1_MIN):.0f}° to {np.degrees(THETA1_MAX):.0f}°')
print(f'Elbow range:    {np.degrees(THETA2_MIN):.0f}° to {np.degrees(THETA2_MAX):.0f}°')

---
## Part 1: The Arm Class — Forward Kinematics

The **elbow position**: $x_e = l_1 \cos(\theta_1),\; y_e = l_1 \sin(\theta_1)$

Note that $\theta_1$ is measured from the world $x$-axis (global), while $\theta_2$ is measured relative to the upper arm (body-fixed). The absolute forearm angle is therefore $(\theta_1 + \theta_2)$.

The **hand position**:
$$x = l_1 \cos(\theta_1) + l_2 \cos(\theta_1 + \theta_2)$$
$$y = l_1 \sin(\theta_1) + l_2 \sin(\theta_1 + \theta_2)$$

### Task 1.1
COMPLETE `forward_kinematics` and `elbow_position`.

In [ ]:
class Arm:
    """2-link planar arm in the horizontal plane (Gribble et al. 1998 parameters)."""
    
    def __init__(self, l1=0.34, l2=0.46):
        self.l1, self.l2 = l1, l2
        self.theta1_lim = (np.radians(0), np.radians(140))
        self.theta2_lim = (np.radians(0), np.radians(145))
    
    def forward_kinematics(self, q):
        """Joint angles [theta1,theta2] → hand position [x,y]."""
        theta1, theta2 = q
        # TODO: COMPLETE (Equations 2.2a, 2.2b)
        x = ...  # your code here
        y = ...  # your code here
        return np.array([x, y])
    
    def elbow_position(self, q):
        """Joint angles → elbow position [x_e, y_e]."""
        theta1, theta2 = q
        # TODO: COMPLETE
        x_e = ...  # your code here
        y_e = ...  # your code here
        return np.array([x_e, y_e])
    
    def is_valid(self, q):
        """Check if q is within anatomical joint limits."""
        t1, t2 = q
        return (self.theta1_lim[0] <= t1 <= self.theta1_lim[1] and
                self.theta2_lim[0] <= t2 <= self.theta2_lim[1])

### Task 1.2 — Verify Your Implementation

In [ ]:
# === Test FK ===
arm = Arm()

p = arm.forward_kinematics([0, 0])
assert np.allclose(p, [0.80, 0.0], atol=1e-10), f"Test 1 failed: {p}"
print(f"✓ Abducted, extended → ({p[0]:.2f}, {p[1]:.2f})")

p = arm.forward_kinematics([np.pi/2, 0])
assert np.allclose(p, [0.0, 0.80], atol=1e-10), f"Test 2 failed: {p}"
print(f"✓ Straight ahead, extended → ({p[0]:.2f}, {p[1]:.2f})")

p = arm.forward_kinematics([0, np.pi/2])
assert np.allclose(p, [0.34, 0.46], atol=1e-10), f"Test 3 failed: {p}"
print(f"✓ Abducted, elbow 90° → ({p[0]:.2f}, {p[1]:.2f})")

assert arm.is_valid([np.radians(140), np.radians(145)])
assert not arm.is_valid([np.radians(150), 0])
assert not arm.is_valid([-0.1, 0])
print("✓ Joint limit checks passed")

print("\n🎉 All FK tests passed!")

### Task 1.3 — Visualize the Arm
COMPLETE `draw_arm` to plot shoulder→elbow→hand.

In [ ]:
def draw_arm(arm, q, ax=None, color='#2E86AB', lw=3):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    shoulder = np.array([0, 0])
    elbow = arm.elbow_position(q)
    hand = arm.forward_kinematics(q)
    
    # TODO: COMPLETE — plot lines (shoulder→elbow→hand) and joint markers
    # your code here
    
    ax.set_xlim(-0.2, 0.9); ax.set_ylim(-0.2, 0.9)
    ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    return ax

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
configs = [
    ([np.radians(0), np.radians(0)], 'Abducted, extended'),
    ([np.radians(45), np.radians(90)], 'Typical reach'),
    ([np.radians(90), np.radians(45)], 'Forward reach'),
    ([np.radians(120), np.radians(100)], 'Across midline'),
]
for ax, (q, title) in zip(axes, configs):
    draw_arm(arm, q, ax=ax); ax.set_title(title)
plt.tight_layout(); plt.show()

---
## Part 2: The Workspace

With anatomical limits ($\theta_1 \in [0°, 140°]$, $\theta_2 \in [0°, 145°]$), the workspace is a **semicircular crescent**.

### Task 2.1
Sample all valid joint configurations and plot the hand positions.

In [ ]:
# TODO: COMPLETE — create arrays within anatomical limits
n = 200
theta1_range = ...  # your code here
theta2_range = ...  # your code here

hand_positions = []
# TODO: COMPLETE — loop, compute FK, store
# your code here

hand_positions = np.array(hand_positions)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(hand_positions[:, 0], hand_positions[:, 1], s=0.5, alpha=0.3, c='#2E86AB')

# Outer arc (elbow extended)
t1_arc = np.linspace(THETA1_MIN, THETA1_MAX, 200)
ax.plot((arm.l1+arm.l2)*np.cos(t1_arc), (arm.l1+arm.l2)*np.sin(t1_arc), 'k--', lw=1.5, label='Outer (θ₂=0°)')

# Inner arc (elbow max flexion)
inner_pts = np.array([arm.forward_kinematics([t1, THETA2_MAX]) for t1 in t1_arc])
ax.plot(inner_pts[:,0], inner_pts[:,1], 'k:', lw=1.5, label='Inner (θ₂=145°)')

ax.plot(0, 0, 'ko', ms=8)
ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Reachable Workspace (Anatomically Constrained)'); ax.legend()
plt.show()
print(f"Max reach: {arm.l1+arm.l2:.2f} m")
r_min = np.sqrt(arm.l1**2 + arm.l2**2 + 2*arm.l1*arm.l2*np.cos(THETA2_MAX))
print(f"Min reach: {r_min:.2f} m")

### Question 2.1
Why is the workspace a crescent rather than a full annulus?

*Your answer here:*


---
## Part 3: Inverse Kinematics

The following equations are derived by expanding the FK equations using angle addition identities and grouping terms (Lecture Notes, Eqs. 2.2–3.8).

### Equations
$$\cos(\theta_2) = \frac{x^2 + y^2 - l_1^2 - l_2^2}{2 l_1 l_2}, \quad \theta_2 = \pm\arccos(c_2)$$
$$k_1 = l_1 + l_2\cos(\theta_2),\; k_2 = l_2\sin(\theta_2),\; \theta_1 = \text{atan2}(y,x) - \text{atan2}(k_2,k_1)$$

**Only the positive $\theta_2$ solution (elbow flexion) is anatomically valid.** The solver filters by joint limits.

### Task 3.1

In [ ]:
def inverse_kinematics(self, p):
    """Hand position → joint angles (anatomically valid only)."""
    x, y = p
    r_sq = x**2 + y**2
    # TODO: COMPLETE — compute cos(theta2)
    c2 = ...  # your code here
    if abs(c2) > 1.0:
        return []
    solutions = []
    for sign in [1, -1]:
        # TODO: COMPLETE — compute theta2, k1, k2, theta1
        theta2 = ...  # your code here
        k1 = ...  # your code here
        k2 = ...  # your code here
        theta1 = ...  # your code here
        q = np.array([theta1, theta2])
        if self.is_valid(q):
            solutions.append(q)
    return solutions

Arm.inverse_kinematics = inverse_kinematics

In [ ]:
# === Test IK ===
arm = Arm()
for q_orig in [[np.radians(30),np.radians(60)],[np.radians(90),np.radians(45)],
               [np.radians(60),np.radians(90)],[np.radians(45),np.radians(120)]]:
    p = arm.forward_kinematics(q_orig)
    sols = arm.inverse_kinematics(p)
    ok = any(np.allclose(p, arm.forward_kinematics(s), atol=1e-8) for s in sols)
    assert ok, f"Failed for q={np.degrees(q_orig).round(1)}°"
    print(f"✓ ({np.degrees(q_orig[0]):.0f}°,{np.degrees(q_orig[1]):.0f}°) → {len(sols)} valid solution(s)")

assert len(arm.inverse_kinematics([5,5])) == 0
print("✓ Unreachable → empty")
for s in arm.inverse_kinematics(arm.forward_kinematics([np.radians(70),np.radians(5)])):
    assert s[1] >= 0
print("✓ No hyperextension")
print("\n🎉 All IK tests passed!")

### Task 3.2 — Visualize IK Solution

In [ ]:
target = np.array([0.3, 0.5])
solutions = arm.inverse_kinematics(target)
fig, ax = plt.subplots(figsize=(8, 8))
for i, q_sol in enumerate(solutions):
    draw_arm(arm, q_sol, ax=ax, color=['#2E86AB','#E8553A'][i%2])
    print(f"Solution {i+1}: θ₁={np.degrees(q_sol[0]):.1f}°, θ₂={np.degrees(q_sol[1]):.1f}°")
ax.plot(*target, 'r*', ms=15, zorder=10); ax.set_title(f'IK: target ({target[0]:.2f}, {target[1]:.2f})')
plt.show()

### Question 3.1
Why does the anatomically constrained model typically yield only **one** valid IK solution?

*Your answer here:*


---
## Part 4: The Jacobian

$$J = \begin{bmatrix} -l_1 s_1 - l_2 s_{12} & -l_2 s_{12} \\ l_1 c_1 + l_2 c_{12} & l_2 c_{12} \end{bmatrix}, \quad \det(J) = l_1 l_2 \sin(\theta_2)$$

Only reachable singularity: $\theta_2 = 0°$ — the two columns of $J$ become parallel, so the hand can only move tangentially (radial motion becomes impossible).

### Task 4.1

In [ ]:
def jacobian(self, q):
    theta1, theta2 = q
    s1, c1 = np.sin(theta1), np.cos(theta1)
    s12, c12 = np.sin(theta1+theta2), np.cos(theta1+theta2)
    # TODO: COMPLETE (Equations 4.3a-d)
    J = np.array([
        [..., ...],
        [..., ...]
    ])
    return J

Arm.jacobian = jacobian

In [ ]:
arm = Arm()
q_test = np.array([np.radians(45), np.radians(60)])
J = arm.jacobian(q_test)
eps = 1e-7
J_num = np.zeros((2,2))
for j in range(2):
    qp, qm = q_test.copy(), q_test.copy()
    qp[j] += eps; qm[j] -= eps
    J_num[:,j] = (arm.forward_kinematics(qp) - arm.forward_kinematics(qm))/(2*eps)
assert np.allclose(J, J_num, atol=1e-5), f"Mismatch:\n{J}\nvs\n{J_num}"
print("✓ Jacobian matches numerical differentiation")
det_J = np.linalg.det(J)
assert np.isclose(det_J, arm.l1*arm.l2*np.sin(q_test[1]), atol=1e-10)
print(f"✓ det(J) = {det_J:.6f}")
assert np.isclose(np.linalg.det(arm.jacobian([np.radians(45),0])), 0, atol=1e-10)
print("✓ Singular at θ₂=0°")
print("\n🎉 All Jacobian tests passed!")

### Question 4.1
At $\theta_2 = 0°$, what direction of hand motion becomes impossible?

*Your answer here:*


---
## Part 5: Manipulability Ellipsoids

$A = JJ^T$. Yoshikawa: $w = |l_1 l_2 \sin(\theta_2)|$. Max at $\theta_2 = 90°$, zero at $\theta_2 = 0°$.

### Task 5.1

In [ ]:
def manipulability_ellipsoid(arm, q):
    J = arm.jacobian(q)
    center = arm.forward_kinematics(q)
    # TODO: COMPLETE — A = J @ J.T, eigendecompose, compute w
    A = ...  # your code here
    eigenvalues, eigenvectors = ...  # your code here
    w = ...  # your code here
    return center, eigenvalues, eigenvectors, w

In [ ]:
def draw_ellipsoid(ax, center, eigenvalues, eigenvectors, scale=1.0, color='#2E86AB', alpha=0.3):
    width = 2*np.sqrt(max(eigenvalues[0],0))*scale
    height = 2*np.sqrt(max(eigenvalues[1],0))*scale
    angle = np.degrees(np.arctan2(eigenvectors[1,0], eigenvectors[0,0]))
    ax.add_patch(Ellipse(xy=center, width=width, height=height, angle=angle,
                         facecolor=color, edgecolor=color, alpha=alpha, lw=1.5))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
for t1 in np.linspace(np.radians(5), np.radians(135), 10):
    for t2 in np.linspace(np.radians(15), np.radians(140), 6):
        c, ev, evc, w = manipulability_ellipsoid(arm, [t1,t2])
        draw_ellipsoid(ax, c, ev, evc, scale=0.5, color=plt.cm.RdYlBu(w/(arm.l1*arm.l2)), alpha=0.5)
ax.plot(0,0,'ko',ms=6); ax.set_aspect('equal')
ax.set_xlim(-0.2,0.9); ax.set_ylim(-0.2,0.9)
ax.set_title('Manipulability Ellipsoids (Blue=high, Red=low)')
plt.show()

### Task 5.2 — Plot $w$ vs $\theta_2$ over $[0°, 145°]$

In [ ]:
# TODO: COMPLETE — plot w vs theta2
# your code here


### Question 5.1
At what $\theta_2$ is $w$ maximized? Why doesn't it depend on $\theta_1$?

*Your answer here:*


---
## Part 6: Challenge Problems (Optional)

### Challenge 6.1: Straight-Line Hand Paths (Morasso 1981)

**Tip:** To ensure all waypoints are reachable, generate your start and end points from known valid joint configurations using `arm.forward_kinematics()`, e.g., `start = arm.forward_kinematics([np.radians(40), np.radians(60)])`.

Generate a straight-line path in hand space between the two points. For each waypoint, solve the IK. Then plot:
1. The hand path in Cartesian space (should be a straight line)
2. The joint angle trajectories θ₁(t) and θ₂(t)

Are the joint angle trajectories straight? Why or why not?

In [ ]:
# your code here


### Challenge 6.2: Singularity Map
Heat map of $|\det(J)|$ across the workspace.

In [ ]:
# your code here


### Challenge 6.3: Force Ellipsoid
Plot $(J^{-T}J^{-1})$ alongside velocity ellipsoid. Verify speed-force tradeoff.

In [ ]:
# your code here


---
## Summary

You built the `Arm` class with: `forward_kinematics`, `elbow_position`, `inverse_kinematics`, `jacobian`, `is_valid`.

**Conventions:** θ₁ ∈ [0°,140°] (shoulder), θ₂ ∈ [0°,145°] (elbow). Workspace = semicircular crescent. Only singularity = θ₂ = 0°.

**Next week:** Muscles (Gribble et al. 1998 model).